# 1.1 - Train Teacher on a Group-Disjoint Split

Re-trains the Task-0 ResNet50 + LSTM, this time on a split where no source video
appears in more than one of train / val / test.

**Why this notebook exists.** UCF101 clips are named `v_Class_gXX_cYY.avi`, and every
`cYY` inside one `gXX` group is cut from the SAME source video. The split this dataset
ships with was made randomly per clip, so 78% of groups have clips in several splits at
once. A test clip's near-duplicate, same scene, same lighting, frames apart, was sitting
in the training set.

Everything measured that way was inflated: the old teacher read 98.25% on base test and
the frozen-feature probe read 97.49%. The giveaway was 1-NN scoring 97.13%, almost the
same as a trained linear probe. An untrained nearest-neighbour matches a trained
classifier when it can retrieve a near-duplicate of the test clip, which is the
signature of leakage rather than of a good representation.

Hyperparameters are copied from `1.0_backbone_transfer.ipynb` unchanged, 5 epochs and
lr 1e-4, so the split is the only variable. The new checkpoint is written next to the
old one, not over it.

## **Connect Colab**

In [ ]:
import os, sys, subprocess
from getpass import getpass

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "embeddings"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [ ]:
os.environ["KAGGLE_USERNAME"] = "colab"
os.environ["KAGGLE_KEY"]      = "xx"

## **Download DATASET**

In [ ]:
from src.bootstrap import setup
setup(drive=True, dataset=True)

In [ ]:
# imports
%run /content/ulal/src/imports.py
set_seed(cfg.SEED)

## **Preprocess data**

Confirm the shipped split is leaky, build a group-disjoint one, verify it, then write
the clips. Only the 10 base classes are needed here.

In [ ]:
leaking = check_group_leakage(cfg.SELECTED_CLASSES)

`build_group_split` pools every video regardless of the folder it shipped in, then
assigns WHOLE groups to one split. UCF101 has 25 groups per class, so 18 / 3 / 4 gives
roughly 72 / 12 / 16 percent. Groups are ordered by name and cut by index, so the split
is deterministic and needs no seed.

In [ ]:
base_split = build_group_split(cfg.SELECTED_CLASSES, train_groups=18, val_groups=3)

In [ ]:
clashes = describe_group_split(base_split, cfg.SELECTED_CLASSES)

Do not run the next cell unless the check above says `OK - group-disjoint`.

In [ ]:
# BASE
cfg.OUTPUT_ROOT = cfg.BASE_ROOT
preprocess_group_split(base_split, cfg.SELECTED_CLASSES)

In [ ]:
cfg.SELECTED_CLASSES

In [ ]:
class_to_idx = {cls_name: i for i, cls_name in enumerate(cfg.SELECTED_CLASSES)}

print(f"class_to_idx:\n{class_to_idx}")

## **Create Loaders**

In [ ]:
TRAIN_DIR = os.path.join(cfg.BASE_ROOT, "train")
VAL_DIR   = os.path.join(cfg.BASE_ROOT, "val")
TEST_DIR  = os.path.join(cfg.BASE_ROOT, "test")

train_dataset = UCF101Clips(TRAIN_DIR, class_to_idx)
val_dataset   = UCF101Clips(VAL_DIR,   class_to_idx)
test_dataset  = UCF101Clips(TEST_DIR,  class_to_idx)

train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=cfg.BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=cfg.BATCH_SIZE, shuffle=False)

In [ ]:
print(f"{'split':<8}{'clips':>8}{'clips/class':>13}")
print("-" * 29)
print(f"{'train':<8}{len(train_dataset):>8}{len(train_dataset)/10:>13.1f}")
print(f"{'val':<8}{len(val_dataset):>8}{len(val_dataset)/10:>13.1f}")
print(f"{'test':<8}{len(test_dataset):>8}{len(test_dataset)/10:>13.1f}")
print("-" * 29)
print(f"{'TOTAL':<8}{len(train_dataset)+len(val_dataset)+len(test_dataset):>8}")

## **Upload Model**

`ResNet50LSTM` is the same architecture `ResNet50LSTMTeacher` loads later, everything
frozen except `layer4`, LSTM 2048 to 256, dropout, then the classifier. Notebook 2.0
converts the `resnet.*` keys to `backbone.*` through `remap_teacher_checkpoint`.

In [ ]:
model_ResNet50 = ResNet50LSTM(
    hidden_size = cfg.TEACHER_HIDDEN,
    num_classes = len(cfg.SELECTED_CLASSES),
    dropout_p   = cfg.DROPOUT_P,
).to(device)

In [ ]:
n_total     = sum(p.numel() for p in model_ResNet50.parameters())
n_trainable = sum(p.numel() for p in model_ResNet50.parameters() if p.requires_grad)

print(f"total     : {n_total:>12,}")
print(f"trainable : {n_trainable:>12,}   ({100*n_trainable/n_total:.0f}%)")
print(f"frozen    : {n_total - n_trainable:>12,}")
print(f"\nfc: in={model_ResNet50.fc.in_features}  out={model_ResNet50.fc.out_features}")

## **Task Zero**

The only stage where the whole backbone is trainable, and correctly so, this is the
transfer-learning step. `train_model` keeps the best epoch by validation accuracy and
writes it to Drive.

The new checkpoint goes to a **new filename**. The old `ResNet50_10C.pth` is left alone
so you can compare, but it should not be used again.

Expect a lower number than the old 98.25%. That is the point: the old one was reading
its own training frames.

In [ ]:
CKPT_PATH = f"{cfg.DRIVE_PROJECT}/resnet50/models/ResNet50_10C_groupsplit.pth"
os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)

EPOCHS = 5
LR     = 1e-4

print(f"saving to: {CKPT_PATH}")

In [ ]:
set_seed(cfg.SEED)

results_resnet_50 = train_model(
    model_ResNet50,
    train_loader,
    val_loader,
    num_epochs = EPOCHS,
    lr         = LR,
    device     = device,
    save_path  = CKPT_PATH,
)

In [ ]:
plot_training_results(results_resnet_50)

If validation accuracy is still climbing at the last epoch, raise `EPOCHS` and run
again. Without leakage the model has to actually learn the classes, so it may need
more epochs than the original 5. `train_model` keeps the best epoch, so extra epochs
cost time but cannot make the checkpoint worse.

## **Evaluation**

The test set now contains only source videos the model has never seen in any form.

In [ ]:
test_acc, all_preds, all_labels = test_model(model_ResNet50, test_loader, device=device)

print(f"\nBase test accuracy (group-disjoint): {test_acc:.2%}")
print(f"Old leaky split reported            : 98.25%")

In [ ]:
plot_confusion_matrix(
    all_labels, all_preds,
    num_classes  = len(cfg.SELECTED_CLASSES),
    classes_list = cfg.SELECTED_CLASSES,
    name         = "Teacher, group-disjoint",
    save_path    = f"{cfg.RESULTS_DIR}/cm_teacher_groupsplit",
)

In [ ]:
print_detailed_metrics(
    all_labels   = all_labels,
    all_preds    = all_preds,
    num_classes  = len(cfg.SELECTED_CLASSES),
    classes_list = cfg.SELECTED_CLASSES,
)

## **Save**

Load the saved file back the exact way notebook 2.0 will, through
`ResNet50LSTMTeacher` and `remap_teacher_checkpoint`, so a key mismatch surfaces here
rather than in the next notebook.

In [ ]:
teacher = ResNet50LSTMTeacher(
    hidden_size = cfg.TEACHER_HIDDEN,
    num_classes = len(cfg.SELECTED_CLASSES),
    dropout_p   = cfg.DROPOUT_P,
).to(device)

mapped = remap_teacher_checkpoint(torch.load(CKPT_PATH, map_location=device))
missing, unexpected = teacher.load_state_dict(mapped, strict=False)

real_missing = [k for k in missing if "num_batches_tracked" not in k]
if real_missing:
    raise RuntimeError(f"Checkpoint missing keys: {real_missing[:5]}")

print(f"missing   : {len(real_missing)}")
print(f"unexpected: {len(unexpected)}")

In [ ]:
teacher.eval()
reload_acc, reload_preds, reload_labels = test_model(ModelWrapper(teacher).to(device), test_loader, device)

print(f"\nas ResNet50LSTM        : {test_acc:.2%}")
print(f"as ResNet50LSTMTeacher : {reload_acc:.2%}")
print("Checkpoint round-trips correctly." if abs(reload_acc - test_acc) < 0.01
      else "MISMATCH - the two architectures disagree, do not continue.")

In [ ]:
summary_1_1 = {
    "stage":            "1.1_teacher_group_split",
    "classes":          cfg.SELECTED_CLASSES,
    "checkpoint":       CKPT_PATH,
    "epochs":           EPOCHS,
    "lr":               LR,
    "train_groups":     18,
    "val_groups":       3,
    "clips":            {"train": len(train_dataset), "val": len(val_dataset), "test": len(test_dataset)},
    "test_acc":         float(test_acc),
    "test_acc_reload":  float(reload_acc),
    "best_val_acc":     float(results_resnet_50["best_val_acc"]),
    "leaky_split_acc":  0.9825,
}

os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
with open(f"{cfg.RESULTS_DIR}/stage1_teacher_groupsplit.json", "w") as f:
    json.dump(summary_1_1, f, indent=2, default=float)

print(f"Saved -> {cfg.RESULTS_DIR}/stage1_teacher_groupsplit.json")

---

**Before running notebook 2.0**, point it at this checkpoint. Either set the env var in
the setup cell:

```python
os.environ["ULAL_RESNET50_PATH"] = "/content/drive/MyDrive/apai/resnet50/models/ResNet50_10C_groupsplit.pth"
```

or edit `RESNET50_PATH` in `src/config/config.py` and push.

Then run 2.0. It rebuilds the cache from the group-disjoint clips and re-measures the
probe ceiling. Expect the ceiling to land well below 97.49%; that lower number is the
honest one, and it is what every notebook-4 result should be read against.